# Imports

In [1]:

from pathlib import Path
from langchain_core.tools import tool
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.subagents import SubAgent
import uuid
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from chroma_tools.tools import index_python_chunk, retrieve_python_knowledge
from copilotkit import CopilotKitMiddleware
from langgraph.checkpoint.memory import MemorySaver



## Configurar el backend de filesystem

In [2]:
backend = FilesystemBackend(
    root_dir=str(Path.cwd()),
    virtual_mode=True
)

## Definir 2 subagentes

In [3]:
subagents = [
    SubAgent(
        name="python_indexer",
        description="Agente encargado de indexar fragmentos de código en Chroma.",
        system_prompt="Eres un indexador técnico. Formatea y guarda usando 'index_python_chunk'.",
        model="openai:gpt-4o-mini",
        tools=[index_python_chunk]
    ),
    SubAgent(
        name="python_retriever",
        description="Agente encargado de buscar conocimiento en Chroma.",
        system_prompt="Busca contexto técnico usando 'retrieve_python_knowledge' antes de responder.",
        model="openai:gpt-4o-mini",
        tools=[retrieve_python_knowledge]
    )
]

## Configuracion de Conversation ID

In [ ]:
# 1. Definir el System Prompt del Orquestador Central
ORCHESTRATOR_SYSTEM_PROMPT = """Eres el Orquestador Central (Master Agent) de un entorno de desarrollo avanzado de Python.
Tu objetivo principal es coordinar la automatización de tareas, el análisis de código y la gestión de una base de datos de conocimiento persistente en Chroma.

Cuentas con herramientas directas y un equipo de subagentes especializados. Debes actuar como un director de orquesta inteligente:

1. GESTIÓN DE CONSULTAS (RETRIEVAL-AGENTIC):
   - Cuando el usuario te haga una pregunta técnica, antes de inventar una respuesta, evalúa si es necesario buscar antecedentes.
   - Puedes invocar la herramienta 'retrieve_python_knowledge' directamente o delegar la tarea en tu subagente 'python_retriever'.
   - Fusiona el contexto recuperado de forma coherente para dar respuestas de calidad de producción.

2. GESTIÓN DE MEMORIA (INDEXACIÓN):
   - Si el usuario te proporciona un fragmento de código valioso, una solución a un bug o un playbook técnico, NO lo dejes pasar.
   - Delega en el subagente 'python_indexer' para que limpie, estructure e indexe esa información usando 'index_python_chunk'.

3. DELEGACIÓN EFICIENTE:
   - No hagas todo el trabajo pesado tú mismo si un subagente está mejor capacitado para ello usa la herramienta task.
   - Supervisa los resultados de los subagentes. Si la respuesta de un subagente es incompleta, pídile correcciones antes de entregar el resultado final al usuario.

Mantén un tono profesional, directo, altamente técnico y enfocado en la eficiencia de código."""

# 2. Inyectarlo en la creación del Deep Agent
agent = create_deep_agent(
    model="openai:gpt-4o-mini",
    backend=backend,
    system_prompt=ORCHESTRATOR_SYSTEM_PROMPT,  # <-- Aquí inyectas el prompt del supervisor
    tools=[index_python_chunk, retrieve_python_knowledge],
    middleware=[CopilotKitMiddleware()],
    subagents=subagents,
    checkpointer=MemorySaver(),
)

### Estructura de Resultados 

In [5]:
# %%
from IPython.display import display, Markdown

def mostrar_respuesta(resultado):
    """
    Extrae el último mensaje del agente del estado del grafo 
    y lo renderiza como Markdown limpio en la libreta.
    """
    # 1. Obtener la lista de mensajes del output
    messages = resultado.get("messages", [])
    if not messages:
        print("No se recibieron mensajes en el resultado.")
        return
    
    # 2. Tomar el último mensaje (la respuesta más reciente)
    ultimo_mensaje = messages[-1]
    
    # 3. Extraer el contenido textual (independientemente de si es un objeto LangChain o un dict)
    content = getattr(ultimo_mensaje, "content", "") or ultimo_mensaje.get("content", "")
    
    # 4. Renderizar visualmente en la celda
    display(Markdown("---"))
    display(Markdown(f"### 🤖 **Respuesta del Orquestador:**\n\n{content}"))
    display(Markdown("---"))

### Invocar el agente

In [ ]:

result = agent.invoke(
    {"messages": [{"role": "user", "content": "sube ese chunk en chroma "}]},
    config={"configurable": {"thread_id": "mi-thread-1"}}
)
# Mostrar el output limpio y estructurado
mostrar_respuesta(result)

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """Sube este chunk a chroma La clase SubAgent (que técnicamente es un TypedDict) define la especificación para crear agentes secundarios síncronos. Contiene campos obligatorios como name, description y system_prompt, junto con campos opcionales para configurar herramientas, modelos, middleware y permisos 1 .

Detalle de la Estructura SubAgent
En el archivo libs/deepagents/deepagents/middleware/subagents.py, SubAgent se define como un TypedDict para permitir una configuración declarativa flexible 2 .

Campos Obligatorios
Estos tres campos son esenciales para que el agente principal sepa qué hace el subagente y cómo invocarlo:
- name: Identificador único usado por la herramienta task 3 .
- description: Explica al LLM cuándo delegar tareas a este subagente 4 .
- system_prompt: Las instrucciones base que guiarán el comportamiento del subagente 5 .

Campos Opcionales de Configuración
Permiten personalizar el entorno de ejecución del subagente:
- tools: Lista de herramientas. Si se omite, hereda las del agente principal 6 .
- model: Permite especificar un modelo diferente (ej. un modelo más ligero para tareas simples) 7 .
- middleware: Lista de middleware adicional específico para este subagente 8 .
- permissions: Reglas de acceso al sistema de archivos que sobrescriben las del padre 9 .
- response_format: Define un esquema (Pydantic o JSON) para obtener respuestas estructuradas 10 .

Otros Tipos Relacionados
- CompiledSubAgent: Para subagentes ya construidos como un Runnable de LangChain/LangGraph 11 .
- AsyncSubAgent: Para agentes que corren de forma remota y asíncrona mediante graph_id 12 .

Uso en el Sistema
Cuando utilizas create_deep_agent, el sistema procesa estas definiciones para:
1. Resolver el modelo y permisos: Si no se definen, se heredan del contexto global 13 .
2. Inyectar Middleware base: Cada SubAgent recibe automáticamente herramientas de sistema como FilesystemMiddleware y SummarizationMiddleware 14 .
3. Habilitar la herramienta task: El SubAgentMiddleware utiliza estas especificaciones para exponer la capacidad de delegación al modelo 15 .

Notes
- TypedDict vs Class: En Python, TypedDict se usa aquí para que puedas definir subagentes como diccionarios simples sin instanciar clases complejas, facilitando la configuración desde archivos YAML o JSON 2 .
- Aislamiento: Los subagentes operan con un estado filtrado para evitar que el historial irrelevante del padre contamine su contexto 16 .

Wiki pages you might want to explore:
- Middleware System (langchain-ai/deepagents)

Citations
File: libs/deepagents/deepagents/middleware/subagents.py (L25-63)
class SubAgent(TypedDict):
    \"\"\"Specification for an agent.
    When using `create_deep_agent`, subagents automatically receive a default middleware
    stack (TodoListMiddleware, FilesystemMiddleware, SummarizationMiddleware, etc.) before
    any custom `middleware` specified in this spec.
    Required fields:
        name: Unique identifier for the subagent.
            The main agent uses this name when calling the `task()` tool.
        description: What this subagent does.
            Be specific and action-oriented. The main agent uses this to decide when to delegate.
        system_prompt: Instructions for the subagent.
            Include tool usage guidance and output format requirements.
    Optional fields:
        tools: Tools the subagent can use.
            If not specified, inherits tools from the main agent via `default_tools`.
        model: Override the main agent's model.
            Use the format `'provider:model-name'` (e.g., `'openai:gpt-4o'`).
        middleware: Additional middleware for custom behavior, logging, or rate limiting.
        interrupt_on: Configure human-in-the-loop for specific tools.
            Requires a checkpointer.
        skills: Skill source paths for SkillsMiddleware.
            List of paths to skill directories (e.g., `["/skills/user/", "/skills/project/"]`).
        permissions: Filesystem permission rules for this subagent.
            If omitted, inherits the parent agent's permissions. If provided,
            replaces the parent agent's rules entirely for this subagent.
            Rules are evaluated in declaration order; the first match wins.
    \"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L65-66)
    name: str
    \"\"\"Unique identifier for the subagent.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L68-69)
    description: str
    \"\"\"What this subagent does. The main agent uses this to decide when to delegate.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L71-72)
    system_prompt: str
    \"\"\"Instructions for the subagent.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L74-75)
    tools: NotRequired[Sequence[BaseTool | Callable | dict[str, Any]]]
    \"\"\"Tools the subagent can use. If not specified, inherits from main agent.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L77-78)
    model: NotRequired[str | BaseChatModel]
    \"\"\"Override the main agent's model. Use `'provider:model-name'` format.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L80-81)
    middleware: NotRequired[list[AgentMiddleware]]
    \"\"\"Additional middleware for custom behavior.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L89-93)
    permissions: NotRequired[list[FilesystemPermission]]
    \"\"\"List of ``FilesystemPermission`` rules for this subagent.
    If omitted, inherits the parent agent's permissions. If specified, replaces
    the parent's permissions entirely for this subagent.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L100-105)
    response_format: NotRequired[ResponseFormat[Any] | type | dict[str, Any]]
    \"\"\"Structured output response format for the subagent.
    When specified, the subagent will produce a `structured_response` conforming to the
    given schema. The structured response is JSON-serialized and returned as the
    ToolMessage content to the parent agent, replacing the default last-message extraction.\"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L137-148)
class CompiledSubAgent(TypedDict):
    \"\"\"A pre-compiled agent spec.
    !!! note
        The runnable's state schema must include a 'messages' key.
        This is required for the subagent to communicate results back to the main agent.
    When the subagent completes, the final message in the 'messages' list will be
    extracted and returned as a `ToolMessage` to the parent agent.
    \"\"\"

File: libs/deepagents/deepagents/middleware/subagents.py (L171-183)
# State keys that are excluded when passing state to subagents and when returning updates from subagents.

File: libs/deepagents/deepagents/middleware/subagents.py (L473-485)
class SubAgentMiddleware(AgentMiddleware[Any, ContextT, ResponseT]):
    \"\"\"Middleware for providing subagents to an agent via a `task` tool.\"\"\"

File: libs/deepagents/deepagents/middleware/async_subagents.py (L34-58)
class AsyncSubAgent(TypedDict):
    \"\"\"Specification for an async subagent running on a remote Agent Protocol server.
    Async subagents connect to any Agent Protocol-compliant server via the LangGraph SDK.
    \"\"\"
    name: str
    description: str
    graph_id: str

File: libs/deepagents/deepagents/graph.py (L522-529)
            raw_subagent_model = spec.get("model", model)
            subagent_model = resolve_model(raw_subagent_model)
            subagent_permissions = spec.get("permissions", permissions)

File: libs/deepagents/deepagents/graph.py (L532-541)
            subagent_middleware: list[AgentMiddleware[Any, Any, Any]] = [
                TodoListMiddleware(),
                FilesystemMiddleware(
                    backend=backend,
                    custom_tool_descriptions=_subagent_profile.tool_description_overrides,
                    _permissions=subagent_permissions,
                ),
                create_summarization_middleware(subagent_model, backend),
                PatchToolCallsMiddleware(),
            ]"""
            }
        ]
    },
    config={"configurable": {"thread_id": "mi-thread-1"}},
)

# Mostrar el output limpio y estructurado
mostrar_respuesta(result)